## 1. 패키지 설치

In [1]:
# 패키지 설치 (Colab/로컬 공통)
%pip install -q langchain langchain-openai python-dotenv gradio

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. API 키 로드

In [2]:
import os
from pathlib import Path

from dotenv import load_dotenv


MODEL_NAME = "gpt-4o-mini"

load_dotenv()
if not os.environ.get("OPENAI_API_KEY"):
    load_dotenv(Path.cwd() / "langchain-web-chatbot" / ".env")

if not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("프로젝트 .env에 OPENAI_API_KEY를 설정하세요.")

print(f"API 키 로드 완료 / 모델: {MODEL_NAME}")


API 키 로드 완료 / 모델: gpt-4o-mini


## 3. 라이브러리 및 환경 설정

In [3]:
# 1-6교시 부품 통합 import
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.chat_history import InMemoryChatMessageHistory
import gradio as gr

chatbot_options = {"type": "messages"} if int(gr.__version__.split(".")[0]) < 6 else {}

print(f"✅ import 완료\nlangchain_openai \nGradio: {gr.__version__}")

c:\workspace\chatbot\langchain-web-chatbot\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ import 완료
langchain_openai 
Gradio: 6.29.0


## 4. LLM과 프롬프트 정의

In [4]:
# LLM 정의
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

# 프롬프트: 튜터 변수 + 이전 대화 자리 + 학습자 질문
prompt = ChatPromptTemplate.from_messages([
    ("system", "{tutor_description}"),       # 튜터가 변수
    MessagesPlaceholder("history"),          # 이전 대화 자리 (4교시)
    ("human", "{question}")
])

# 기본 체인
chain = prompt | llm

print("✅ 체인 구성 완료")

✅ 체인 구성 완료


## 5. 대화 세션 저장소 구성

In [5]:
# 모든 세션의 학습 기록을 담는 사전
store = {}

def get_session_history(session_id: str):
    """session_id에 해당하는 히스토리를 가져오거나 새로 만들기"""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]

print("✅ 세션 저장소 준비 완료")

✅ 세션 저장소 준비 완료


## 6. 대화 메모리 체인 구성

In [6]:
# 메모리 체인 (4교시 4부 패턴 그대로)
# 메모리 wrapper로 감싸기
chain_with_memory = RunnableWithMessageHistory(
    chain,                              # 감쌀 체인
    get_session_history,                # 히스토리 가져올 함수
    input_messages_key="question",      # 사용자 입력 변수 이름 ({question})
    history_messages_key="history",     # MessagesPlaceholder 이름
)

print("✅ 메모리 체인 구성 완료")

✅ 메모리 체인 구성 완료


c:\workspace\chatbot\langchain-web-chatbot\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3823: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


## 7. 튜터 데이터 정의 및 JSON 저장

In [7]:
import json

# 튜터 사전: 이름 → system 메시지
TUTORS = {
    "친절한 입문자 튜터": {
        "system": "1. 당신은 친절한 입문자 전용 학습 튜터입니다. 2. 어려운 개념도 일상적인 비유와 예시로 쉽게 설명하세요. 3. 전문 용어가 나오면 풀어서 말하고, 학습자가 이해했는지 확인 질문을 한 번씩 던지세요.",
        "temperature": 0.3,
    },
    "코드 실습 튜터": {
        "system": "당신은 프로그래밍·데이터 실습 튜터입니다. 모든 답변에 실행 가능한 짧은 Python 코드 예제를 포함하세요. 이론보다 직접 해보는 방식을 우선하며, 코드는 주석과 함께 제공하세요.",
        "temperature": 0.2,
    },
    "이론 정밀 튜터": {
        "system": "당신은 학술적 정확성을 중시하는 이론 튜터입니다. 정의를 정확하게 제시하고, 개념 사이의 관계를 명확히 구분해서 설명하세요. 모호한 표현이나 과한 비유는 자제하세요.",
        "temperature": 0.0,
    },
    "시험 대비 튜터": {
        "system": "당신은 시험 합격을 돕는 튜터입니다. 핵심 요점을 번호 매겨 정리하고, 자주 출제되는 함정 포인트와 예상 문제를 함께 제시하세요.",
        "temperature": 0.4,
    },
    "소크라테스식 튜터": {
        "system": "당신은 답을 바로 알려주지 않는 튜터입니다. 학습자가 스스로 답에 도달하도록 단계별 질문을 던지며 안내하세요. 다만 학습자가 막혀 보이면 적절한 힌트를 주세요.",
        "temperature": 0.8,
    },
    "훈련교관식 튜터": {
        "system": "당신은 빡센 특수부대 훈련교관식 튜터입니다. 학습자가 답을 틀리면 바로 피드백을 주고하면서, 거친 말과 함께 다그치듯 학습시킵니다. 정확한 답을 제시하며 반복 학습을 유도하세요.",
        "temperature": 1.0,
    },
}
print(f"✅ {len(TUTORS)}명의 튜터 등록: {list(TUTORS.keys())}")

TUTORS_PATH = "tutors.json"

if not os.path.exists(TUTORS_PATH):
    with open(TUTORS_PATH, "w", encoding="utf-8") as f:
        json.dump(TUTORS, f, ensure_ascii=False, indent=2)
    print("tutors.json 파일을 만들었습니다.")

✅ 6명의 튜터 등록: ['친절한 입문자 튜터', '코드 실습 튜터', '이론 정밀 튜터', '시험 대비 튜터', '소크라테스식 튜터', '훈련교관식 튜터']


## 8. 튜터 설정 JSON 불러오기

In [8]:
# 7교시 도전 과제 C: open + json.load로 튜터 설정 불러오기
with open(TUTORS_PATH, encoding="utf-8") as f:
    TUTORS = json.load(f)

print(f"✅ {len(TUTORS)}명의 튜터 설정 로드 완료")


✅ 7명의 튜터 설정 로드 완료


## 9. 학습 주제 요약


In [9]:
# 7교시 도전 과제 A의 메모리 순회 + 과제 D의 주제 요약 호출
def summarize_topic(message, tutor_name):
    """메모리에 저장된 대화와 새 질문으로 현재 학습 주제를 요약"""
    session_id = f"tutor_{tutor_name}"
    lines = []
    if session_id in store:
        for m in store[session_id].messages:
            role = "학습자" if m.type == "human" else "튜터"
            lines.append(f"[{role}] {m.content}")
    lines.append(f"[학습자] {message}")

    result = llm.invoke([
        {"role": "system", "content": "지금까지 대화의 학습 주제를 한국어로 20자 이내로 요약하세요. 주제만 출력하세요."},
        {"role": "user", "content": "\n".join(lines)},
    ])
    return result.content.strip()


## 10. Gold 튜터별 스트리밍 응답 처리


In [10]:
def respond_stream(message, history, tutor_name, current_topic, tutor_system):
    if not message:
        yield "", history, current_topic
        return

    history = history + [{"role": "user", "content": message}]
    history = history + [{"role": "assistant", "content": ""}]
    yield "", history, current_topic

    current_topic = summarize_topic(message, tutor_name)
    yield "", history, current_topic

    # 7교시 도전 과제 B: 선택한 튜터의 temperature를 llm.bind로 적용
    tutor = TUTORS[tutor_name]
    tutor_chain_with_memory = RunnableWithMessageHistory(
        prompt | llm.bind(temperature=tutor["temperature"]),
        get_session_history,
        input_messages_key="question",
        history_messages_key="history",
    )

    # 7교시 Gold: 튜터별 히스토리 객체 사용
    session_id = f"tutor_{tutor_name}"
    full_response = ""
    for chunk in tutor_chain_with_memory.stream(
        {"question": message, "tutor_description": tutor_system},
        config={"configurable": {"session_id": session_id}}
    ):
        full_response += chunk.content
        history[-1]["content"] = full_response
        yield "", history, current_topic

## 11. 대화 초기화 및 저장 기능

In [11]:
# 7교시 Gold: 튜터 변경 시 해당 튜터의 히스토리 복원
def load_tutor_history(tutor_name):
    session_id = f"tutor_{tutor_name}"
    messages = []
    if session_id in store:
        for m in store[session_id].messages:
            role = "user" if m.type == "human" else "assistant"
            messages.append({"role": role, "content": m.content})
    return TUTORS[tutor_name]["system"], messages, ""


def clear_chat(tutor_name):
    """선택한 튜터의 화면과 메모리만 초기화"""
    session_id = f"tutor_{tutor_name}"
    if session_id in store:
        store[session_id].clear()
    return [], ""


def save_log(tutor_name):
    """선택한 튜터의 학습 기록만 텍스트로 저장"""
    session_id = f"tutor_{tutor_name}"
    if session_id not in store or not store[session_id].messages:
        return "저장할 학습 내용이 없습니다."
    lines = [f"[튜터] {tutor_name}"]
    for m in store[session_id].messages:
        role = "학습자" if m.type == "human" else "튜터"
        lines.append(f"[{role}] {m.content}")

    # 튜터 이름에 파일명 제한 문자가 있어도 저장 가능하도록 목록 순번 사용
    tutor_number = list(TUTORS.keys()).index(tutor_name) + 1
    filename = f"learning_tutor{tutor_number}_{datetime.now().strftime('%Y%m%d_%H%M%S_%f')}.txt"
    with open(filename, "w", encoding="utf-8") as f:
        f.write("\n\n".join(lines))
    return f"✅ {filename}에 저장됨"


## 12. UI에서 튜터 추가 및 설정 저장


In [12]:
# 튜터 JSON 저장 기능: 입력 검증, 안전한 파일 교체, Gradio 결과 갱신
import json
import math
import os
import tempfile
from pathlib import Path
from threading import Lock

tutor_save_lock = Lock()


def select_tutor(tutor_name):
    description, history, topic = load_tutor_history(tutor_name)
    return tutor_name, description, TUTORS[tutor_name]["temperature"], history, topic


def persist_tutor(tutor_name, tutor_description, temperature):
    """튜터 설정을 저장하고 정리된 이름과 추가/수정 여부를 반환."""
    if not isinstance(tutor_name, str) or not tutor_name.strip():
        raise ValueError("튜터 스타일 이름을 입력하세요.")
    if not isinstance(tutor_description, str) or not tutor_description.strip():
        raise ValueError("튜터 설명을 입력하세요.")
    if (
        isinstance(temperature, bool)
        or not isinstance(temperature, (int, float))
        or not math.isfinite(temperature)
        or not 0 <= temperature <= 2
    ):
        raise ValueError("Temperature는 0부터 2 사이의 숫자여야 합니다.")

    name = tutor_name.strip()
    settings = {
        "system": tutor_description.strip(),
        "temperature": float(temperature),
    }
    path = Path(TUTORS_PATH).resolve()

    # 같은 앱에서 동시에 저장해도 읽기/쓰기 순서가 겹치지 않도록 보호
    with tutor_save_lock:
        if path.exists():
            try:
                with path.open(encoding="utf-8") as f:
                    updated_tutors = json.load(f)
            except json.JSONDecodeError as exc:
                raise ValueError("튜터 JSON 파일이 손상되었습니다. 파일 내용을 수정한 뒤 다시 저장하세요.") from exc
            if not isinstance(updated_tutors, dict):
                raise ValueError("튜터 JSON의 최상위 값은 이름을 키로 하는 객체여야 합니다.")
        else:
            updated_tutors = dict(TUTORS)

        action = "수정" if name in updated_tutors else "추가"
        updated_tutors[name] = settings
        temporary_path = None
        try:
            # 같은 폴더의 임시 파일에 먼저 기록해 기존 JSON의 부분 저장 방지
            with tempfile.NamedTemporaryFile(
                mode="w", encoding="utf-8", dir=path.parent,
                prefix=f".{path.name}.", suffix=".tmp", delete=False,
            ) as f:
                temporary_path = Path(f.name)
                json.dump(updated_tutors, f, ensure_ascii=False, indent=2, allow_nan=False)
                f.write("\n")
                f.flush()
                os.fsync(f.fileno())
            os.replace(temporary_path, path)
        finally:
            if temporary_path is not None:
                temporary_path.unlink(missing_ok=True)

        # 파일 교체 성공 후에만 실행 중인 설정 반영; 대화 store는 유지
        TUTORS.clear()
        TUTORS.update(updated_tutors)

    return name, action


def save_tutor(tutor_name, tutor_description, temperature):
    """저장 버튼 콜백: 설정 저장과 화면 갱신을 연결."""
    try:
        name, action = persist_tutor(tutor_name, tutor_description, temperature)
    except ValueError as exc:
        return (*[gr.update() for _ in range(6)], f"❌ {exc}")
    except OSError:
        return (
            *[gr.update() for _ in range(6)],
            "❌ 튜터 파일을 저장하지 못했습니다. 저장 폴더와 쓰기 권한을 확인하세요.",
        )

    description, history, topic = load_tutor_history(name)
    return (
        gr.update(choices=list(TUTORS.keys()), value=name),
        name, description, TUTORS[name]["temperature"], history, topic,
        f"✅ '{name}' 튜터를 {action}했습니다. ({TUTORS_PATH})",
    )


## 13. Gradio 챗봇 UI 실행

In [13]:
# Gold 챗봇 UI (Blocks 전체)
gr.close_all()

with gr.Blocks(title="AI 학습 튜터") as demo_gold:
    gr.Markdown("# 📚 AI 학습 튜터 (Gold)")
    gr.Markdown("튜터마다 별개의 학습 기록을 유지합니다. 튜터를 바꾸면 이전 대화가 복원됩니다.")

    with gr.Row():
        # 좌측 패널: 튜터 선택 + 설명 + 초기화
        with gr.Column(scale=1):
            tutor_dd = gr.Dropdown(choices=list(TUTORS.keys()),
                                    value="친절한 입문자 튜터",
                                    label="튜터 스타일")
            tutor_name_input = gr.Textbox(value="친절한 입문자 튜터",
                                          label="튜터 스타일 이름",
                                          placeholder="새 이름을 입력하면 새 튜터로 추가됩니다")
            tutor_desc = gr.Textbox(value=TUTORS["친절한 입문자 튜터"]["system"],
                                     interactive=True, lines=5,
                                     label="튜터 설명")
            tutor_temperature = gr.Slider(minimum=0, maximum=2, step=0.1,
                                           value=TUTORS["친절한 입문자 튜터"]["temperature"],
                                           label="Temperature")
            save_tutor_btn = gr.Button("💾 튜터 설정 저장", variant="primary")
            tutor_save_status = gr.Textbox(label="튜터 저장 결과", interactive=False)
            gr.Markdown("새 이름으로 저장하면 튜터가 추가됩니다. 기존 이름으로 저장하면 설정이 수정됩니다.")
            clear_btn = gr.Button("🧹 현재 튜터 학습 기록 초기화")
            save_btn = gr.Button("💾 학습 내용 저장")

        # 우측 패널: 현재 주제와 채팅창
        with gr.Column(scale=2):
            topic = gr.Textbox(label="현재 학습 주제", interactive=False,
                               lines=1, max_lines=1, placeholder="대화 주제가 여기에 표시됩니다")
            chatbot = gr.Chatbot(height=420, **chatbot_options)
            with gr.Row():
                msg = gr.Textbox(placeholder="질문 입력...", scale=4, container=False)
                send = gr.Button("전송", variant="primary", scale=1)

    # 이벤트 핸들러 (같은 with 블록 안에서 정의)
    tutor_dd.change(fn=select_tutor, inputs=tutor_dd,
                    outputs=[tutor_name_input, tutor_desc, tutor_temperature, chatbot, topic])
    save_tutor_btn.click(fn=save_tutor,
                         inputs=[tutor_name_input, tutor_desc, tutor_temperature],
                         outputs=[tutor_dd, tutor_name_input, tutor_desc, tutor_temperature,
                                  chatbot, topic, tutor_save_status])
    send.click(respond_stream, [msg, chatbot, tutor_dd, topic, tutor_desc], [msg, chatbot, topic])
    msg.submit(respond_stream, [msg, chatbot, tutor_dd, topic, tutor_desc], [msg, chatbot, topic])
    clear_btn.click(fn=clear_chat, inputs=tutor_dd, outputs=[chatbot, topic])
    save_btn.click(fn=save_log, inputs=tutor_dd, outputs=gr.Textbox())

demo_gold.launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
